# 📊 Week 1: Correlation vs Causation

**CS 2700: Causal Inference** | Utah Valley University | Spring 2026

---

## 🎯 What You'll Learn

In this notebook, you will **observe real data analysis** and answer questions to deepen your understanding:

| Concept | What You'll See |
|---------|-----------------|
| Spurious Correlation | Ice cream & drowning data analysis |
| Confounding | How controlling for variables removes fake correlations |
| Simpson's Paradox | The famous Berkeley admissions case |
| Counterfactuals | "What if" reasoning with data |

### ⚠️ No Coding Required!

**Simply:**
1. **Run each code cell** (click ▶️ or press Shift+Enter)
2. **Observe the output** (tables, graphs, numbers)
3. **Answer the questions** in the ✏️ boxes

---

## ⚙️ Setup: Run This First

Click the cell below and press **Shift+Enter** to load the required tools.

In [ ]:
# Run this cell first (Shift+Enter)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
print("✅ Setup complete! You're ready to begin.")

---

# Part 1: The Ice Cream & Drowning Paradox 🍦

From your reading, you learned that ice cream sales and drowning deaths are **correlated**. But does eating ice cream *cause* drowning? 

Let's see this in real data!

## 1.1 The Data

**Run the cell below** to see monthly data on temperature, ice cream sales, and drowning deaths.

In [ ]:
# Create monthly data (run this cell)
np.random.seed(42)

ice_cream_data = pd.DataFrame({
    'month': ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
              'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
    'temperature': [35, 38, 48, 58, 68, 78, 85, 82, 72, 60, 48, 38],
})

# Temperature affects both ice cream sales and drowning
ice_cream_data['ice_cream_sales'] = ice_cream_data['temperature'] * 10 + np.random.normal(0, 50, 12)
ice_cream_data['drowning_deaths'] = ice_cream_data['temperature'] * 0.5 + np.random.normal(0, 3, 12)

print("📊 MONTHLY DATA")
print("=" * 60)
print(ice_cream_data.to_string(index=False))

# Calculate correlation
correlation = ice_cream_data['ice_cream_sales'].corr(ice_cream_data['drowning_deaths'])
print(f"\n🔢 Correlation between Ice Cream Sales and Drowning: r = {correlation:.3f}")
print("\n💡 A correlation near 1.0 is VERY strong!")

---

### ✏️ Question 1.1

Look at the data table above.

**a)** In which months are ice cream sales highest? In which months are drowning deaths highest?

**b)** What do you notice about temperature in those same months?

---

*Double-click this cell to type your answer:*

**Your Answer:**

a) 

b) 

---

## 1.2 Visualizing the Relationships

**Run the cell below** to see three scatter plots that reveal what's really going on.

In [ ]:
# Visualize the relationships (run this cell)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: The MISLEADING correlation
axes[0].scatter(ice_cream_data['ice_cream_sales'], ice_cream_data['drowning_deaths'], 
                s=100, alpha=0.7, c='red')
axes[0].set_xlabel('Ice Cream Sales ($)', fontsize=11)
axes[0].set_ylabel('Drowning Deaths', fontsize=11)
r = ice_cream_data['ice_cream_sales'].corr(ice_cream_data['drowning_deaths'])
axes[0].set_title(f'❌ MISLEADING: r = {r:.2f}', fontsize=12, color='red')

# Plot 2: Temperature → Ice Cream
axes[1].scatter(ice_cream_data['temperature'], ice_cream_data['ice_cream_sales'], 
                s=100, alpha=0.7, c='orange')
axes[1].set_xlabel('Temperature (°F)', fontsize=11)
axes[1].set_ylabel('Ice Cream Sales ($)', fontsize=11)
r1 = ice_cream_data['temperature'].corr(ice_cream_data['ice_cream_sales'])
axes[1].set_title(f'✅ Temperature → Ice Cream: r = {r1:.2f}', fontsize=12, color='green')

# Plot 3: Temperature → Drowning
axes[2].scatter(ice_cream_data['temperature'], ice_cream_data['drowning_deaths'], 
                s=100, alpha=0.7, c='blue')
axes[2].set_xlabel('Temperature (°F)', fontsize=11)
axes[2].set_ylabel('Drowning Deaths', fontsize=11)
r2 = ice_cream_data['temperature'].corr(ice_cream_data['drowning_deaths'])
axes[2].set_title(f'✅ Temperature → Drowning: r = {r2:.2f}', fontsize=12, color='green')

plt.tight_layout()
plt.show()

print("\n🧠 KEY INSIGHT: Temperature is the CONFOUNDER!")
print("   It causes BOTH ice cream sales AND drowning deaths.")

---

### ✏️ Question 1.2

Look at the three graphs above.

**a)** The LEFT graph shows ice cream vs drowning. If you ONLY saw this graph, what might you incorrectly conclude?

**b)** The MIDDLE and RIGHT graphs show temperature's effect. In your own words, explain why ice cream and drowning are correlated even though one doesn't cause the other.

---

*Double-click this cell to type your answer:*

**Your Answer:**

a) 

b) 

---

## 1.3 Controlling for the Confounder

Now let's see what happens when we **remove** the effect of temperature.

**Run the cell below** to see the correlation AFTER controlling for temperature.

In [ ]:
# Control for temperature (run this cell)
from scipy import stats

# Remove temperature's effect from both variables
slope1, int1, _, _, _ = stats.linregress(ice_cream_data['temperature'], ice_cream_data['ice_cream_sales'])
ice_residuals = ice_cream_data['ice_cream_sales'] - (slope1 * ice_cream_data['temperature'] + int1)

slope2, int2, _, _, _ = stats.linregress(ice_cream_data['temperature'], ice_cream_data['drowning_deaths'])
drown_residuals = ice_cream_data['drowning_deaths'] - (slope2 * ice_cream_data['temperature'] + int2)

# New correlation after controlling
original_r = ice_cream_data['ice_cream_sales'].corr(ice_cream_data['drowning_deaths'])
controlled_r = np.corrcoef(ice_residuals, drown_residuals)[0, 1]

print("📊 CONTROLLING FOR TEMPERATURE")
print("=" * 50)
print(f"   BEFORE controlling: r = {original_r:.3f} (strong!)")
print(f"   AFTER controlling:  r = {controlled_r:.3f} (almost zero!)")
print("\n✅ The correlation DISAPPEARS when we control for temperature!")
print("\n💡 This proves: Ice cream does NOT cause drowning.")
print("   The original correlation was SPURIOUS (fake).")

---

### ✏️ Question 1.3

**a)** What happened to the correlation after we controlled for temperature?

**b)** If someone proposed banning ice cream sales to reduce drowning deaths, what would you tell them?

**c)** From your reading, a confounder must meet 3 criteria. Verify that Temperature meets all three:
   - Does temperature affect ice cream sales? 
   - Does temperature affect drowning?
   - Is temperature caused by ice cream sales?

---

*Double-click this cell to type your answer:*

**Your Answer:**

a) 

b) 

c) 

---

---

# Part 2: Simpson's Paradox 🔄

Now let's explore one of the most famous examples of confounding: the **UC Berkeley Admissions case** (1973).

Berkeley was sued for gender discrimination. The data showed men were admitted at higher rates than women. **But was this really discrimination?**

## 2.1 The Overall Picture

**Run the cell below** to see the admission rates.

In [ ]:
# Berkeley admissions data (run this cell)
data = {
    'department': ['A', 'A', 'B', 'B', 'C', 'C', 'D', 'D', 'E', 'E', 'F', 'F'],
    'gender': ['Male', 'Female', 'Male', 'Female', 'Male', 'Female', 
               'Male', 'Female', 'Male', 'Female', 'Male', 'Female'],
    'admitted': [512, 89, 353, 17, 120, 202, 138, 131, 53, 94, 22, 24],
    'rejected': [313, 19, 207, 8, 205, 391, 279, 244, 138, 299, 351, 317],
}
df = pd.DataFrame(data)
df['total'] = df['admitted'] + df['rejected']
df['admit_rate'] = df['admitted'] / df['total']

male = df[df['gender'] == 'Male']
female = df[df['gender'] == 'Female']

male_rate = male['admitted'].sum() / male['total'].sum()
female_rate = female['admitted'].sum() / female['total'].sum()

print("📊 UC BERKELEY ADMISSIONS (1973) - OVERALL")
print("=" * 50)
print(f"   Men admitted:   {male_rate:.1%}")
print(f"   Women admitted: {female_rate:.1%}")
print(f"   Gap:            {(male_rate - female_rate)*100:.1f} percentage points")
print("\n⚠️ This looks like discrimination against women!")

---

### ✏️ Question 2.1

Based on the overall rates above:

**a)** Who has a higher admission rate?

**b)** If you were a lawyer suing Berkeley for discrimination, would this data support your case?

---

*Double-click this cell to type your answer:*

**Your Answer:**

a) 

b) 

---

## 2.2 Breaking Down by Department

**Run the cell below** to see admission rates for EACH department.

In [ ]:
# Rates by department (run this cell)
pivot = df.pivot(index='department', columns='gender', values='admit_rate').round(3)
pivot['Higher Rate'] = pivot.apply(
    lambda x: '👩 Women' if x['Female'] > x['Male'] else '👨 Men', axis=1
)

print("📊 ADMISSION RATES BY DEPARTMENT")
print("=" * 60)
print(pivot)
print("\n" + "=" * 60)
women_higher = (pivot['Higher Rate'] == '👩 Women').sum()
print(f"\n✅ SURPRISE: Women have HIGHER rates in {women_higher} out of 6 departments!")

---

### ✏️ Question 2.2

Compare the overall rates to the department-level rates.

**a)** In how many departments do women have a higher admission rate than men?

**b)** This is a paradox! Overall data shows men doing better, but department data shows women doing better. Write down your initial guess for why this might happen.

---

*Double-click this cell to type your answer:*

**Your Answer:**

a) 

b) 

---

## 2.3 Visualizing the Paradox

**Run the cell below** to see the paradox as charts.

In [ ]:
# Visualize Simpson's Paradox (run this cell)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall
axes[0].bar(['Men', 'Women'], [male_rate, female_rate], color=['#3498db', '#e74c3c'], edgecolor='black')
axes[0].set_ylabel('Admission Rate', fontsize=12)
axes[0].set_title('❌ OVERALL: Looks Like Men Are Favored', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 0.6)
for i, v in enumerate([male_rate, female_rate]):
    axes[0].text(i, v + 0.02, f'{v:.1%}', ha='center', fontsize=14, fontweight='bold')

# By department
x = np.arange(6)
width = 0.35
pivot_vals = df.pivot(index='department', columns='gender', values='admit_rate')
axes[1].bar(x - width/2, pivot_vals['Male'].values, width, label='Men', color='#3498db', edgecolor='black')
axes[1].bar(x + width/2, pivot_vals['Female'].values, width, label='Women', color='#e74c3c', edgecolor='black')
axes[1].set_xlabel('Department', fontsize=12)
axes[1].set_ylabel('Admission Rate', fontsize=12)
axes[1].set_title('✅ BY DEPARTMENT: Women Often Higher!', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['A', 'B', 'C', 'D', 'E', 'F'])
axes[1].legend()
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

print("\n🧠 THIS IS SIMPSON'S PARADOX!")
print("   The trend REVERSES when you look at subgroups.")

## 2.4 Finding the Confounder

So what caused this paradox? Let's look at **where** men and women applied.

**Run the cell below** to see application patterns.

In [ ]:
# Application patterns (run this cell)
apps = df.pivot(index='department', columns='gender', values='total')
dept_totals = df.groupby('department').agg({'admitted': 'sum', 'total': 'sum'})
apps['Dept Rate'] = (dept_totals['admitted'] / dept_totals['total']).round(2)
apps['% Female'] = ((apps['Female'] / (apps['Male'] + apps['Female'])) * 100).round(1)
apps = apps.sort_values('Dept Rate', ascending=False)

print("📊 WHERE DID APPLICANTS APPLY?")
print("=" * 70)
print(apps[['Male', 'Female', '% Female', 'Dept Rate']])
print("\n💡 KEY INSIGHT:")
print("   • Dept A & B: EASY to get in (high rates) → FEW women applied")
print("   • Dept C-F: HARD to get in (low rates) → MORE women applied")
print("\n   DEPARTMENT CHOICE is the CONFOUNDER!")

---

### ✏️ Question 2.3

Look at where men and women applied.

**a)** Which departments have the highest admission rates? Which gender applied more to those?

**b)** Which departments have the lowest admission rates? Which gender applied more to those?

**c)** Now explain Simpson's Paradox in your own words: Why did the overall data show men with higher rates, even though women had higher rates in most departments?

---

*Double-click this cell to type your answer:*

**Your Answer:**

a) 

b) 

c) 

---

## 2.5 The Counterfactual

From your reading, **counterfactuals** are "what would have happened" under different circumstances.

**Run the cell below** to answer: "What if women had applied to departments like men did?"

In [ ]:
# Counterfactual analysis (run this cell)

# How did men distribute their applications?
male_apps = df[df['gender'] == 'Male'].set_index('department')['total']
male_distribution = male_apps / male_apps.sum()

# Female admission rates by department
female_rates = df[df['gender'] == 'Female'].set_index('department')['admit_rate']

# If women applied like men, what would their rate be?
adjusted_female_rate = (female_rates * male_distribution).sum()

print("📊 COUNTERFACTUAL: 'What if women applied like men?'")
print("=" * 60)
print(f"   Actual female rate:      {female_rate:.1%}")
print(f"   Counterfactual rate:     {adjusted_female_rate:.1%}")
print(f"   Male rate:               {male_rate:.1%}")
print("\n" + "=" * 60)
print(f"\n✅ After controlling for department choice:")
print(f"   Women ({adjusted_female_rate:.1%}) > Men ({male_rate:.1%})")
print("\n🎓 CONCLUSION: No discrimination against women.")
print("   The gap was due to different application patterns.")

---

### ✏️ Question 2.4

**a)** After controlling for department choice, which gender has the higher admission rate?

**b)** What is the "counterfactual" we estimated? (Describe it in plain English)

**c)** Based on this analysis, was Berkeley discriminating against women? Explain your reasoning.

---

*Double-click this cell to type your answer:*

**Your Answer:**

a) 

b) 

c) 

---

---

# Part 3: Connecting to Your Reading 📖

Now let's make sure you understand the key concepts.

### ✏️ Question 3.1: Types of Research Questions

From Chapter 2, classify each question:

| Question | Type (Descriptive / Predictive / Causal) |
|----------|------------------------------------------|
| "What % of applicants were admitted in 1973?" | |
| "Will this student likely be admitted?" | |
| "Did being female CAUSE lower admission rates?" | |

---

*Double-click this cell to type your answers:*

**Your Answers:**

1. 

2. 

3. 

---

### ✏️ Question 3.2: Three Explanations for Correlation

From Chapter 4, when X and Y are correlated, there are three explanations. For each, identify the most likely:

| Correlation | Explanation | Why? |
|-------------|-------------|------|
| Ice cream ↔ Drowning | | |
| Hours studied ↔ Exam score | | |
| Firefighters at scene ↔ Fire damage | | |

---

*Double-click this cell to type your answers:*

**Your Answers:**

1. Ice cream ↔ Drowning: 

2. Hours studied ↔ Exam score: 

3. Firefighters ↔ Fire damage: 

---

### ✏️ Question 3.3: The Fundamental Problem

From Chapter 4, we can never observe both what happened AND what would have happened for the same person.

For a female applicant who was rejected:
- **Factual:** She applied as female and was rejected
- **Counterfactual:** What would have happened if she applied as male?

**a)** Why can't we observe this counterfactual for any individual?

**b)** How did we ESTIMATE the counterfactual in our Berkeley analysis?

---

*Double-click this cell to type your answers:*

**Your Answers:**

a) 

b) 

---

---

# ✅ Summary

| Concept | What You Observed |
|---------|-------------------|
| **Correlation ≠ Causation** | Ice cream and drowning: r = 0.9, but no causal link |
| **Confounder** | Temperature causes both; Department choice causes both |
| **Controlling** | After removing confounder effect, fake correlations disappear |
| **Simpson's Paradox** | Overall trend reversed when looking at subgroups |
| **Counterfactual** | "What if women applied like men?" |

---

# 📤 Submission

Before submitting:

- [ ] Ran all code cells
- [ ] Answered ALL ✏️ questions
- [ ] Saved the notebook
- [ ] Submitted to Canvas

| Section | Points |
|---------|--------|
| Part 1: Ice Cream & Drowning (Q1.1-1.3) | 30% |
| Part 2: Simpson's Paradox (Q2.1-2.4) | 40% |
| Part 3: Reading Concepts (Q3.1-3.3) | 30% |

---

**🔜 Next Week:** The Potential Outcomes Framework — Y(1) and Y(0)!